# Controlling MODFLOW 6 with the API — F. Sequential reactions (denitrification)

This is part F of the MODFLOW 6 API series — see [**A. Basic usage**](modflow-api-A.ipynb) for the API lifecycle and [**C. Adjusting recharge with a callback**](modflow-api-C.ipynb) for the callback mechanism this example builds on.

Use the API to add *chemistry* that MODFLOW 6 does not solve on its own. A single simulation holds one groundwater-flow (GWF) model and **three** groundwater-transport (GWT) models — one per nitrogen species — wired to flow with GWF-GWT exchanges and sharing a single transport solution. At each transport time step a `modflowapi` callback applies a first-order sequential denitrification chain

$$\mathrm{NO_3} \;\xrightarrow{\;k_1\;}\; \mathrm{NO_2} \;\xrightarrow{\;k_2\;}\; \mathrm{N_2}$$

by reading each species' concentration from MODFLOW's memory and writing the reaction mass rates back. A lake is the nitrate source; nitrite and dinitrogen start at zero everywhere, so their plumes are *purely reaction products*.

The base flow-and-transport model is the MODFLOW 6 example `ex-gwt-prudic2004t2` (Prudic and others, 2004), which couples SFR (streams), LAK (a lake), and MVR (a mover) to advanced-package transport (SFT/LKT/MVT).

Run the imports and library-location cells below first.

## Imports and setup

Import FloPy, the `modflowapi` interface, and the supporting plotting and data libraries, then create a directory for the figures saved by this notebook.

In [ ]:
%matplotlib inline

import math
import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import numpy as np
import prudic_model as pm
from flopy.plot.styles import styles
from matplotlib.colors import ListedColormap, LogNorm
from matplotlib.ticker import FuncFormatter, StrMethodFormatter
from modflowapi import Callbacks, run_simulation
from notebook_helpers import find_mf6_libraries

In [ ]:
fig_path = pl.Path(".figures")
fig_path.mkdir(exist_ok=True)

### Locate the MODFLOW 6 library

The API drives MODFLOW 6 through its compiled shared library (`libmf6`), not the `mf6` command-line executable. The `find_mf6_libraries` helper in `notebook_helpers.py` locates the library (`.so` on Linux, `.dylib` on macOS, `.dll` on Windows) and the `mf6` executable inside the active pixi environment; confirm that both exist.

In [ ]:
# libmf6 (the shared library the API drives) and the mf6 executable,
# located in the active pixi environment
lib_name, mf6_exe = find_mf6_libraries()

In [ ]:
lib_name.is_file(), mf6_exe.is_file()

## The denitrification problem

Set the model workspace and the transport timing, then pick the reaction-coupling method. The grid, flow, and transport *properties* all come from `ex-gwt-prudic2004t2` and are held fixed in the imported `prudic_model` module — here you set only what the lesson varies. The flow model is steady state; transport runs for ~25 years, split into `nstp_transport` time steps, and the API callback fires once per step.

Pick the **reaction-coupling method** here with `reaction_method`:

- `"src_rhs"` — the callback computes the reaction mass rates and writes them into a Mass Source Loading (SRC) package for each species. SRC adds the terms to the solution right-hand side *and* books them in the GWT budget, so every per-model mass budget closes.
- `"operator_split"` — the callback overwrites each concentration array in place at the end of the time step (operator splitting). Simpler, but the reacted mass shows up as a per-model budget discrepancy because it is not a recognized package term.

In [ ]:
sim_name = "nitrate-nitrite"

# API run workspace (under the git-ignored models/ directory)
workspace = pl.Path(f"models/{sim_name}_api_F")

# reaction-coupling method: "src_rhs" (clean budget) or "operator_split"
reaction_method = "src_rhs"

# transport timing: flow is steady; transport runs ~25 years in nstp_transport
# equal steps, and the API callback fires once per step
total_time = 9131.0  # days (~25 years)
nstp_transport = 300

### Reaction parameters

The chain is two first-order reactions, `NO3 -> NO2 -> N2`, each set by a half-life. The mass yields (`y1`, `y2`) come from molar stoichiometry, so the mass that leaves a parent species is added to its daughter exactly (times the molar-mass ratio).

The lake nitrate concentration follows a *source-reduction scenario*: held at `no3_source_conc` until year 10, then reduced linearly to `no3_source_conc_final` (below the drinking-water limit) by year 15, and held there afterward. Only the larger lake supplies nitrate; the smaller lake is kept at zero.

In [ ]:
SUSTAINED_SOURCE = True  # True: lake held at a (time-varying) NO3; False: slug
no3_source_conc = 50.0  # initial lake nitrate concentration (mg/L)

# source-reduction time series: constant until year 10, reduced to just below the
# nitrate MCL by year 15, then held there
no3_source_conc_final = 10.0
source_reduce_start = 10.0 * 365.0  # days (10 years)
source_reduce_end = 15.0 * 365.0  # days (15 years)
source_ts = [
    (0.0, no3_source_conc),
    (source_reduce_start, no3_source_conc),
    (source_reduce_end, no3_source_conc_final),
    (total_time, no3_source_conc_final),
]

half_life_no3 = 730.0  # NO3 first-order half-life (days)
half_life_no2 = 365.0  # NO2 first-order half-life (days)
k1 = math.log(2.0) / half_life_no3  # NO3 -> NO2 rate (1/day)
k2 = math.log(2.0) / half_life_no2  # NO2 -> N2  rate (1/day)

# species molar masses (g/mol) and the mass yields from molar stoichiometry
M_NO3 = 62.004
M_NO2 = 46.005
M_N2 = 28.014
y1 = M_NO2 / M_NO3  # 1 mol NO3 -> 1 mol NO2  (~0.742 mass)
y2 = 0.5 * M_N2 / M_NO2  # 1 mol NO2 -> 0.5 mol N2 (~0.305 mass)

SPECIES = ["no3", "no2", "n2"]

# map display: mask concentrations below MASK_BELOW (log color scale) and draw the
# drinking-water maximum contaminant level (MCL) contour where one exists. EPA MCLs
# are 10 mg/L as N for nitrate (= 44.3 mg/L as NO3) and 1 mg/L as N for nitrite
# (= 3.28 mg/L as NO2); dinitrogen has none.
MASK_BELOW = 0.01
MCL = {"no3": 44.3, "no2": 3.28}

### Build the base model

The grid, flow and transport properties, boundary and stream data, and the coupled simulation itself all live in the imported `prudic_model` module — this is the base example `ex-gwt-prudic2004t2`, not the API lesson, so it is kept out of the notebook. Call `prudic_model.build_prudic_simulation()` with the species list (`SPECIES`) and the reaction settings from above.

It assembles one `MFSimulation` holding the GWF flow model and one GWT model per species, each registered with a shared transport solution and wired to flow with a GWF-GWT exchange, so the callback sees all three species in the same solution. When `reaction_method == "src_rhs"`, every species also gets a Mass Source Loading (SRC) package with one entry per active cell that the callback overwrites each step; `operator_split` adds none.

In [ ]:
sim = pm.build_prudic_simulation(
    workspace,
    mf6_exe,
    SPECIES,
    reaction_method=reaction_method,
    source_species="no3",
    source_time_series=source_ts if SUSTAINED_SOURCE else None,
    slug_source_conc=no3_source_conc,
    total_time=total_time,
    nstp=nstp_transport,
)
sim.write_simulation(silent=True)
print(f"reaction method: {reaction_method}")
print("wrote simulation to", workspace)

## Apply the reaction with an API callback

The reaction chain is not a MODFLOW package — the API supplies it. Two callbacks are defined below; `reaction_method` (set above) selects which one runs.

- **`SrcRhsCallback`** (`src_rhs`) fires at the *start* of each time step. It reads the current aquifer concentrations, computes the first-order mass rates `r1 = k1·[NO3]·Vp` and `r2 = k2·[NO2]·Vp` (with `Vp` the saturated pore volume of each cell), and writes them into each species' SRC loading array (`SMASSRATE`). MODFLOW then adds those terms to the solution and books them in the budget, so mass transfer between species is exact and every per-model budget closes.
- **`OperatorSplitCallback`** (`operator_split`) fires at the *end* of each time step and decays the concentration arrays in place: `NO3` loses `1 - e^{-k1·dt}` of its mass, `NO2` gains the nitrate yield and loses its own, and `N2` gains the nitrite yield. Simpler, but the reacted mass is invisible to the budget.

The GWT solution vector `X` is laid out as `[aquifer cells | LKT lakes | SFT reaches]`; the reaction is applied only to the aquifer cells (the first `naq` entries). The `src_rhs` callback needs the per-cell saturated pore volume `Vp`, and the results plot needs each species' total dissolved mass over time — both come from `prudic_model` (`saturated_pore_volume_reduced` and `total_mass_in_storage`).

In [ ]:
class OperatorSplitCallback:
    """Apply the NO3->NO2->N2 chain by overwriting the state arrays X at the end
    of each transport time step (operator splitting)."""

    def __init__(self):
        self.history = []
        self.elapsed = 0.0
        self._ptrs = None

    def __call__(self, sim, step):
        if step != Callbacks.timestep_end:
            return
        models = {m.name.lower(): m for m in sim.models}
        if not all(s in models for s in SPECIES):
            return
        if self._ptrs is None:
            mf6 = models["no3"].mf6
            self._ptrs = {
                s: mf6.get_value_ptr(mf6.get_var_address("X", models[s].name))
                for s in SPECIES
            }
            # X is [aquifer cells | LKT lakes | SFT reaches]; react the aquifer only
            self.naq = mf6.get_value(mf6.get_var_address("X", "FLOW")).shape[0]
        c, naq = self._ptrs, self.naq
        dt = sim.delt
        d1 = np.maximum(c["no3"][:naq], 0.0) * (1.0 - math.exp(-k1 * dt))  # NO3 lost
        d2 = np.maximum(c["no2"][:naq], 0.0) * (1.0 - math.exp(-k2 * dt))  # NO2 lost
        # in-place updates write straight into MF6 memory (do not rebind)
        c["no3"][:naq] -= d1
        c["no2"][:naq] += y1 * d1 - d2
        c["n2"][:naq] += y2 * d2
        self._record(dt, c, naq)

    def _record(self, dt, c, naq):
        self.elapsed += dt
        self.history.append((self.elapsed, *(float(c[s][:naq].sum()) for s in SPECIES)))


class SrcRhsCallback:
    """Compute the reaction mass rates from the current concentrations and write
    them into each species' SRC package at the start of each time step. The SRC
    terms enter the solution RHS and are booked in the GWT budget, so every
    per-model mass budget closes."""

    def __init__(self, cell_pore_volume):
        self.pv = cell_pore_volume  # porosity*volume per active node (reduced)
        self.history = []
        self.elapsed = 0.0
        self._x = None  # live concentration views
        self._smr = None  # live SRC SMASSRATE views
        self._red = None  # SRC entry index -> reduced node
        self._sat_addr = None

    def __call__(self, sim, step):
        if step != Callbacks.timestep_start:
            return
        models = {m.name.lower(): m for m in sim.models}
        if not all(s in models for s in SPECIES):
            return
        mf6 = models["no3"].mf6
        if self._x is None:
            self._x = {
                s: mf6.get_value_ptr(mf6.get_var_address("X", models[s].name))
                for s in SPECIES
            }
            # SMASSRATE (not BOUND) is the loading array MF6 uses to formulate the
            # SRC contribution to the solution
            self._smr = {
                s: mf6.get_value_ptr(
                    mf6.get_var_address("SMASSRATE", models[s].name, "SRC-1")
                )
                for s in SPECIES
            }
            # SRC entries were created in reduced-node order, but map explicitly
            # via NODELIST to be safe
            self._red = {
                s: mf6.get_value(
                    mf6.get_var_address("NODELIST", models[s].name, "SRC-1")
                ).astype(int)
                - 1
                for s in SPECIES
            }
            self._sat_addr = mf6.get_var_address("SAT", "FLOW", "NPF")
            # aquifer node count (X also carries LKT/SFT unknowns past this)
            self.naq = self.pv.shape[0]
            assert self._x["no3"][: self.naq].shape == self.pv.shape

        naq = self.naq
        sat = mf6.get_value(self._sat_addr)
        satpv = self.pv * sat  # saturated pore volume
        c = {s: self._x[s][:naq] for s in SPECIES}  # aquifer concentrations
        # first-order reaction mass rates (mass / time) per active node
        r1 = k1 * np.maximum(c["no3"], 0.0) * satpv  # NO3 -> NO2
        r2 = k2 * np.maximum(c["no2"], 0.0) * satpv  # NO2 -> N2
        rate = {
            "no3": -r1,  # nitrate consumed
            "no2": y1 * r1 - r2,  # nitrite produced then consumed
            "n2": y2 * r2,  # dinitrogen produced
        }
        for s in SPECIES:
            self._smr[s][:] = rate[s][self._red[s]]
        self._record(sim.delt, c)

    def _record(self, dt, c):
        self.elapsed += dt
        self.history.append((self.elapsed, *(float(c[s].sum()) for s in SPECIES)))

### Run the model through the API

Create the callback that matches `reaction_method` and hand it to `run_simulation`, which drives the compiled library through the whole simulation, calling back at every time step so the reaction is applied as the transport solution advances.

In [ ]:
if reaction_method == "src_rhs":
    callback = SrcRhsCallback(pm.saturated_pore_volume_reduced(sim))
else:
    callback = OperatorSplitCallback()

run_simulation(lib_name, workspace, callback, verbose=False)

### Check the mass budgets

Read the final-time-step percent discrepancy from each species' listing file. With `src_rhs` the reaction is a booked SRC term, so every per-model budget should close to a tiny fraction of a percent. With `operator_split` the reacted mass is *not* a package term, so each budget shows a discrepancy roughly equal to the mass converted to or from that species.

In [ ]:
print(f"Per-model mass-balance discrepancy (last time step), method={reaction_method}:")
for s in SPECIES:
    pct = None
    for line in (workspace / f"{s}.lst").read_text().splitlines():
        if "PERCENT DISCREPANCY" in line:
            try:
                pct = float(line.split("=")[-1])
            except ValueError:
                pass
    print(f"  {s}: percent discrepancy = {pct}")

### Concentration maps

Map each species at year 10 (the end of the sustained-source period) across four model layers. Concentrations below `MASK_BELOW` are masked and the color scale is logarithmic; the white contour marks the drinking-water MCL where one exists. The magenta star in panel A is the observation cell used for the breakthrough curves below.

In [ ]:
decades = [0.01, 0.1, 1.0, 10.0, 100.0]
plain = FuncFormatter(lambda v, _: f"{v:g}")

lakibd = pm.load_grid_data()[2]
gwf = sim.get_model("flow")
labels = {
    "no3": "Nitrate (NO$_3$)",
    "no2": "Nitrite (NO$_2$)",
    "n2": "Dinitrogen (N$_2$)",
}
map_layers = [0, 2, 4, 7]

# observation cell (layer 1, row 18, col 18), 0-based
obs = (0, 17, 17)
xc = gwf.modelgrid.xcellcenters[obs[1], obs[2]]
yc = gwf.modelgrid.ycellcenters[obs[1], obs[2]]
lake_cmap = ListedColormap(["#6ca6cd"])
lake_overlay = np.ma.masked_where(lakibd == 0, np.ones((pm.nrow, pm.ncol)))

# maps are shown at 10 years (closest transport output time)
_mt = np.array(sim.get_model("no3").output.concentration().times)
map_totim = float(_mt[np.argmin(np.abs(_mt - 10.0 * 365.0))])
map_year = map_totim / 365.0

for s in SPECIES:
    gwt = sim.get_model(s)
    conc = gwt.output.concentration().get_data(totim=map_totim)
    lakec = gwt.lak.output.concentration().get_data(totim=map_totim).flatten()
    il, jl = np.where(lakibd > 0)
    for i, j in zip(il, jl):
        conc[0, i, j] = lakec[lakibd[i, j] - 1]
    cdisp = np.ma.masked_where((conc >= 1.0e29) | (conc < MASK_BELOW), conc)
    vmax = max(float(conc[conc < 1.0e29].max()), MASK_BELOW * 10.0)
    norm = LogNorm(vmin=MASK_BELOW, vmax=vmax)
    with styles.USGSMap():
        fig, axs = plt.subplots(2, 2, figsize=(7, 9), dpi=150, constrained_layout=True)
        for ip, ilay in enumerate(map_layers):
            ax = axs.flatten()[ip]
            pmv = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=ilay)
            pmv.plot_inactive(color_noflow="0.8")
            if ilay == 0:
                pmv.plot_array(lake_overlay, cmap=lake_cmap, vmin=0, vmax=1, alpha=0.6)
            ca = pmv.plot_array(cdisp, norm=norm)
            pmv.plot_bc("CHD-1", color="royalblue")
            if s in MCL:
                cs = pmv.contour_array(
                    conc,
                    masked_values=[1.0e30],
                    colors="white",
                    linewidths=1.5,
                    levels=[MCL[s]],
                )
                ax.clabel(cs, fmt=f"{MCL[s]:g}", fontsize=7, colors="white")
            if ip == 0:
                ax.plot(
                    xc,
                    yc,
                    marker="*",
                    ms=11,
                    mfc="magenta",
                    mec="k",
                    mew=0.6,
                    zorder=6,
                    clip_on=False,
                )
                ax.annotate(
                    "(1, 18, 18)",
                    xy=(xc, yc),
                    xytext=(xc - 3400.0, yc + 2200.0),
                    textcoords="data",
                    ha="right",
                    va="center",
                    fontsize=7,
                    color="black",
                    zorder=7,
                    arrowprops=dict(
                        arrowstyle="-", color="black", lw=0.6, shrinkA=0, shrinkB=4
                    ),
                )
            styles.heading(
                ax=ax, letter=chr(ord("A") + ip), heading=f"Model layer {ilay + 1}"
            )
            ax.set_aspect("equal")
            ax.set_xlabel("x position (ft)")
            ax.set_ylabel("y position (ft)")
            ticks = [t for t in decades if MASK_BELOW * 0.999 <= t <= vmax * 1.001]
            cb = fig.colorbar(ca, ax=ax, shrink=0.5, ticks=ticks)
            cb.ax.yaxis.set_major_formatter(plain)
            cb.ax.minorticks_off()
        mcl_note = f", white = MCL {MCL[s]:g} mg/L" if s in MCL else ""
        fig.suptitle(
            f"{labels[s]} at {map_year:.0f} years (blue = constant head{mcl_note})",
            fontsize=11,
            fontweight="bold",
        )
        fig.savefig(fig_path / f"nitrate-nitrite-conc-map-{s}.png", dpi=150)

### Breakthrough and mass in storage

Two panels: (A) the concentration of each species over time at the observation cell (log scale, so the smaller nitrite and dinitrogen curves are visible), and (B) the total dissolved mass of each species in the aquifer. Nitrate arrives first; nitrite and dinitrogen build up behind it as the reaction proceeds, and all three respond after year 10 when the lake source is reduced.

In [ ]:
colors = {"no3": "tab:blue", "no2": "tab:red", "n2": "tab:green"}
times, mass = pm.total_mass_in_storage(sim, SPECIES)

with styles.USGSPlot():
    fig, (axa, axb) = plt.subplots(
        2, 1, sharex=True, figsize=(7, 4.5), dpi=150, constrained_layout=True
    )
    ymax = 0.0
    for s in SPECIES:
        cobj = sim.get_model(s).output.concentration()
        t = np.array(cobj.times) / 365.0
        series = cobj.get_alldata()[:, obs[0], obs[1], obs[2]]
        axa.plot(t, series, color=colors[s], label=labels[s])
        ymax = max(ymax, float(np.max(series)))
    axa.set_yscale("log")
    axa.set_ylim(0.01, 1.5 * ymax)
    axa.yaxis.set_major_formatter(plain)
    axa.set_ylabel("concentration (mg/L)")
    styles.heading(ax=axa, letter="A", heading="Concentration at cell (1, 18, 18)")
    for s in SPECIES:
        axb.plot(times, mass[s], color=colors[s], label=labels[s])
    styles.graph_legend(ax=axb)
    axb.set_ylim(bottom=0.0)
    axb.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    axb.set_ylabel("Total mass in storage (kg)")
    styles.heading(ax=axb, letter="B", heading="Mass in aquifer storage")
    axb.set_xlim(0.0, 25.0)
    axb.set_xlabel("time (years)")
    fig.align_ylabels((axa, axb))
    fig.savefig(fig_path / "nitrate-nitrite-timeseries.png", dpi=150)

**Recap.** This example used the API for something MODFLOW 6 will not do on its own: couple three transport models through a first-order reaction chain. The `src_rhs` callback wrote reaction mass rates into a SRC package so the terms are booked in every budget, while `operator_split` overwrote the concentration state directly — the same lifecycle and the same `get_value` / `get_value_ptr` memory access from the earlier notebooks, now used to inject chemistry between transport time steps. Switch `reaction_method` to `"operator_split"` and re-run from the build cell to see the reaction as a budget discrepancy instead.